In [1]:
import pandas as pd
import os

In [ ]:
import os
os.chdir(os.path.join('..', 'data', 'educacao'))  # trabalha dentro da pasta de dados

In [3]:
#DIMINUIR NOME

parte_remover = "Censo 2022 - "

pasta = "."
arquivos = [f for f in os.listdir(pasta) 
           if os.path.isfile(f) and not f.endswith('.ipynb')]

print("Renomeando arquivos...")
for nome_arquivo in arquivos:
    novo_nome = nome_arquivo.replace(parte_remover, "")
    if novo_nome != nome_arquivo:
        os.rename(nome_arquivo, novo_nome)
        print(f"{nome_arquivo} → {novo_nome}")

print("Concluído!")

Renomeando arquivos...
Censo 2022 - Alfabetização.xlsx → Alfabetização.xlsx
Censo 2022 - Alfabetização de indígenas.xlsx → Alfabetização de indígenas.xlsx
Censo 2022 - Alfabetização de moradores por espécie de domicílio.xlsx → Alfabetização de moradores por espécie de domicílio.xlsx
Censo 2022 - Nível de instrução.xlsx → Nível de instrução.xlsx
Censo 2022 - Número médio de anos de estudo, por cor ou raça.xlsx → Número médio de anos de estudo, por cor ou raça.xlsx
Censo 2022 - Número médio de anos de estudo, por sexo.xlsx → Número médio de anos de estudo, por sexo.xlsx
Censo 2022 - Pessoas com nível superior completo, por área de formação.xlsx → Pessoas com nível superior completo, por área de formação.xlsx
Censo 2022 - Porcentagem de alfabetizados na população total e em favelas.xlsx → Porcentagem de alfabetizados na população total e em favelas.xlsx
Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade.xlsx → Taxa bruta de frequência escolar, por grupo de idade.xlsx
Censo 

In [6]:
coluna_remover = "Recorte Geográfico"

for arquivo in [f for f in os.listdir(".") if f.endswith('.xlsx')]:
    try:
        df = pd.read_excel(arquivo)
        if coluna_remover in df.columns:
            df.drop(columns=[coluna_remover]).to_excel(arquivo, index=False)
            print(f"✅ {arquivo}: coluna '{coluna_remover}' removida")
    except Exception as e:
        print(f"❌ {arquivo}: {e}")

In [23]:
#SABER AS COLUNAS E OS TIPOS DE DADOS

for arquivo in [f for f in os.listdir(".") if f.endswith('.xlsx')]:
    print(f"\n{arquivo}:")
    df = pd.read_excel(arquivo, nrows=2)
    for coluna in df.columns:
        valor = df[coluna].iloc[1] if len(df) > 1 else "N/A"
        print(f"  {coluna}: {type(valor).__name__}")


Alfabetização.xlsx:
  Situação: str
  População(pessoas): int64
  Percentual: int64

Alfabetização de indígenas.xlsx:
  Situação: str
  População(pessoas): int64
  Percentual: float64

Nível de instrução.xlsx:
  Sem instrução e fundamental incompleto: str
  Fundamental completo e médio incompleto: str
  Médio completo e superior incompleto: str
  Superior completo: str

Número médio de anos de estudo, por cor ou raça.xlsx:
  Cor ou raça: str
  Número médio de anos de estudo: float64

Número médio de anos de estudo, por sexo.xlsx:
  Sexo: str
  Número médio de anos de estudo: float64

Pessoas com nível superior completo, por área de formação.xlsx:
  Área de formação: str
  Pessoas com nível superior completo: int64

Porcentagem de alfabetizados na população total e em favelas.xlsx:
  Indicador: str
  Valor: float64

Taxa bruta de frequência escolar, por grupo de idade.xlsx:
  Grupo de idade: str
  Taxa bruta de frequência escolar (%): float64

Taxa de alfabetização por grupos de idade.

In [17]:
print("🔄 CONVERTENDO MÚLTIPLAS COLUNAS")
print("=" * 50)

# Lista todos os arquivos Excel
arquivos_excel = [f for f in os.listdir(".") if f.endswith('.xlsx')]

print(f"📁 Arquivos que serão processados ({len(arquivos_excel)}):")
for arquivo in arquivos_excel:
    print(f"  • {arquivo}")

# Colunas para converter para FLOAT (percentuais)
colunas_float = [
    "Alfabetização de moradores em domicílios improvisados (%)",
    "Alfabetização de moradores em domicílios coletivos (%)", 
    "Taxa bruta de frequência escolar (%)",
    "Taxa de alfabetização",
    "Valor",
    "Sem instrução e fundamental incompleto",
    "Fundamental completo e médio incompleto",
    "Médio completo e superior incompleto",
    "Superior completo",
    "Número médio de anos de estudo",
    "Percentual"
]

print(f"\n🎯 Ações:")
print("  • Colunas para FLOAT (remover '%'):")
for coluna in colunas_float:
    print(f"    - {coluna}")
print("  • Colunas para INT:")

# Confirmação
confirmar = input("\n❓ Deseja continuar? (s/n): ")

if confirmar.lower() == 's':
    for arquivo in arquivos_excel:
        try:
            print(f"\n📊 Processando: {arquivo}")
            df = pd.read_excel(arquivo)
            modificacoes = []
            
            # Converter colunas para FLOAT (removendo %)
            for coluna in colunas_float:
                if coluna in df.columns:
                    # Backup dos valores originais
                    valores_originais = df[coluna].head(3).tolist()
                    
                    # Remover % e converter para float
                    df[coluna] = (
                        df[coluna]
                        .astype(str)
                        .str.replace('%', '', regex=False)
                        .str.replace(',', '.', regex=False)
                        .str.strip()
                    )
                    df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
                    
                    # Valores após conversão
                    valores_convertidos = df[coluna].head(3).tolist()
                    modificacoes.append(f"✅ {coluna}: float")
                    
                    print(f"   🔄 {coluna}")
                    print(f"      Antes: {valores_originais}")
                    print(f"      Depois: {valores_convertidos}")
        
            # Salvar apenas se houve modificações
            if modificacoes:
                df.to_excel(arquivo, index=False)
                print(f"   💾 ARQUIVO SALVO - Modificações: {', '.join(modificacoes)}")
            else:
                print(f"   ⚠️  Nenhuma coluna especificada encontrada neste arquivo")
                
        except Exception as e:
            print(f"   ❌ ERRO: {e}")
    
    print(f"\n🎉 CONCLUÍDO! Todos os {len(arquivos_excel)} arquivos foram processados!")
else:
    print("❌ Operação cancelada pelo usuário")

🔄 CONVERTENDO MÚLTIPLAS COLUNAS
📁 Arquivos que serão processados (11):
  • Alfabetização.xlsx
  • Alfabetização de indígenas.xlsx
  • Nível de instrução.xlsx
  • Número médio de anos de estudo, por cor ou raça.xlsx
  • Número médio de anos de estudo, por sexo.xlsx
  • Pessoas com nível superior completo, por área de formação.xlsx
  • Porcentagem de alfabetizados na população total e em favelas.xlsx
  • Taxa bruta de frequência escolar, por grupo de idade.xlsx
  • Taxa de alfabetização por grupos de idade.xlsx
  • Taxa de alfabetização por sexo e cor ou raça.xlsx
  • Alfabetização de moradores por espécie de domicílio.xlsx

🎯 Ações:
  • Colunas para FLOAT (remover '%'):
    - Alfabetização de moradores em domicílios improvisados (%)
    - Alfabetização de moradores em domicílios coletivos (%)
    - Taxa bruta de frequência escolar (%)
    - Taxa de alfabetização
    - Valor
    - Sem instrução e fundamental incompleto
    - Fundamental completo e médio incompleto
    - Médio completo e 